In [ ]:
from pathlib import Path
import importlib
import sys

# Resolve project root robustly whether notebook runs from project root or notebooks/
PROJECT_ROOT = (
    Path.cwd() if (Path.cwd() / "config" / "config.yaml").exists() else Path.cwd().parent
).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_module = importlib.import_module("data.load")
preprocess_module = importlib.import_module("data.preprocess")
health_index_module = importlib.import_module("features.health_index")
velocity_module = importlib.import_module("features.velocity")
variability_module = importlib.import_module("features.variability")
clustering_module = importlib.import_module("model.clustering")
risk_module = importlib.import_module("model.risk")
validation_module = importlib.import_module("evaluation.validation")
validation_module = importlib.reload(validation_module)

load_dataset = load_module.load_dataset
load_config = load_module.load_config
preprocess_train = preprocess_module.preprocess_train
preprocess_test = preprocess_module.preprocess_test
build_health_index = health_index_module.build_health_index
build_velocity = velocity_module.build_velocity
build_variability = variability_module.build_variability
build_clustering = clustering_module.build_clustering
build_risk_score = risk_module.build_risk_score
run_validation = validation_module.run_validation

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(PROJECT_ROOT / "data" / "raw")

train_raw, test_raw, _ = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)
train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, velocity_artifacts = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)
train_rs, test_rs, risk_art = build_risk_score(train_cl, test_cl, cl_art)

# Run validation
report = run_validation(train_rs, config=config)
report.print_report()

# Hard assertions
assert report.pct_monotonic_hi > 80.0, \
    f"Too many non-monotonic engines: {report.pct_monotonic_hi:.1f}%"
assert report.pct_valid_cluster > 70.0, \
    f"Too many invalid cluster progressions: {report.pct_valid_cluster:.1f}%"
assert report.risk_rul_pearson_r < report.weak_risk_rul_correlation_threshold, \
    f"Risk-RUL correlation too weak: {report.risk_rul_pearson_r:.4f}"

print("\n✅ All validation assertions passed")